In [ ]:
%pip install cobra

In [2]:
import cobra
import numpy as np
import pandas as pd
from cobra.io import load_model
from cobra.io import load_json_model, save_json_model, load_matlab_model, save_matlab_model, read_sbml_model, write_sbml_model
from cobra import Model, Reaction, Metabolite, Gene

In [6]:
model = read_sbml_model('C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\10-19 Research\\11 Data\\11.09 Models\\Manual_curation\\Energy_consumption\\250203_ETC_complex_IV_update.sbml')

## NGAM calculation

In [68]:
reaction = model.reactions.get_by_id('EX_co2_e')
reaction.bounds = -100.0, 0.0

reaction = model.reactions.get_by_id('EX_o2_e')
reaction.bounds = -100.0, 0.0

# set reactions bounds for a specific reaction 
reaction = model.reactions.get_by_id('EX_h2_e')
reaction.bounds = -15.27, -15.27

In [69]:
reaction = model.reactions.get_by_id('EX_h2_e')
reaction.bounds = -15.27, -15.27

In [13]:
model.objective = 'Growth'
solution = model.optimize()
print(f"Flux through ATPM: {solution.objective_value} mmol/gDW/hr")

Flux through ATPM: 0.06319004284167716 mmol/gDW/hr


GAM determination

In [12]:
reaction = model.reactions.get_by_id('ATPM')
reaction.bounds = 11.55, 1000.0

In [15]:
model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.0003129,0,0.00%
cl_e,EX_cl_e,0.0003129,0,0.00%
co2_e,EX_co2_e,3.018,1,100.00%
cobalt2_e,EX_cobalt2_e,1.517E-06,0,0.00%
cu2_e,EX_cu2_e,0.0004262,0,0.00%
fe2_e,EX_fe2_e,0.0009153,0,0.00%
h2_e,EX_h2_e,19.4,0,0.00%
k_e,EX_k_e,0.01173,0,0.00%
mg2_e,EX_mg2_e,0.0005215,0,0.00%
mn2_e,EX_mn2_e,4.158E-05,0,0.00%


In [72]:
reaction = model.reactions.get_by_id('EX_co2_e')
reaction.bounds = -100.0, 0.0

reaction = model.reactions.get_by_id('EX_o2_e')
reaction.bounds = -100.0, 0.0


In [66]:
list_h2_fluxes=[18,19]
dict_slope={}

for gam in range(1,30):


    GAM_metabolites_1 = {'atp_c': -gam, 'h2o_c': -gam}
    GAM_metabolites_2 = {'adp_c': gam, 'pi_c': gam, 'h_c': gam}

    # Get the biomass reaction
    reaction = model.reactions.get_by_id('Growth')

    # Update the coefficients for GAM_metabolites_1
    for element, new_coefficient in GAM_metabolites_1.items():
        metabolite = model.metabolites.get_by_id(element)
        if metabolite in reaction.metabolites:
            reaction.add_metabolites({metabolite: new_coefficient - reaction.metabolites[metabolite]})
    
    

    # Update the coefficients for GAM_metabolites_2
    for element, new_coefficient in GAM_metabolites_2.items():
        metabolite = model.metabolites.get_by_id(element)
        if metabolite in reaction.metabolites:
            reaction.add_metabolites({metabolite: new_coefficient - reaction.metabolites[metabolite]})
    list_growth=[]

    # set reactions bounds for a specific reaction 
    for fluxe in list_h2_fluxes:
        reaction = model.reactions.get_by_id('EX_h2_e')
        reaction.bounds = -fluxe, 0.0
        model.objective = 'Growth'
        solution = model.slim_optimize()
        list_growth.append(solution)
        print(solution)
    
    dict_slope[gam]=(list_h2_fluxes[0]-list_h2_fluxes[1])/(list_growth[0]-list_growth[1])

print(dict_slope)


0.06914166192587129
0.07376137207682044
0.06892937277712688
0.07353489879787112
0.0687183832423727
0.07330981196681127
0.06850868142386493
0.07308609889093862
0.06830025556863437
0.07286374703202346
0.06809309406630788
0.07264274400392284
0.06788718544695646
0.07242307757036007
0.06768251837895006
0.07220473564258076
0.06747908166691849
0.07198770627718139
0.06727686424967555
0.07177197767392424
0.06707585519824819
0.0715575381736346
0.06687604371386337
0.07134437625599425
0.06667741912606337
0.0711324805376037
0.06647997089076073
0.07092183976987437
0.06628368858840475
0.07071244283707395
0.06608856192213651
0.07050427875435207
0.0658945807159913
0.07029733666583451
0.06570173491312248
0.07009160584273019
0.0655100145740695
0.06988707568146807
0.06531940987505554
0.06968373570189706
0.06512991110629392
0.06948157554546742
0.06494150867035617
0.06928058497349539
0.06475419308054252
0.06908075386543078
0.06456795495928316
0.06888207221714145
0.0643827850365908
0.06868453013925843
0.0641

In [67]:
reaction_id_to_check = 'Growth'
reaction = model.reactions.get_by_id(reaction_id_to_check)
print(f"Reaction ID: {reaction.id}")
print(f"Name: {reaction.name}")
print(f"Equation: {reaction.reaction}")
print(f"Lower Bound: {reaction.lower_bound}")
print(f"Upper Bound: {reaction.upper_bound}")
sum = 0
#reaction.name = '(S)-2-Acetolactate pyruvate-lyase (carboxylating)'
print("\nMetabolites and Stoichiometry:")
for metabolite, coefficient in reaction.metabolites.items():
    print(f"{metabolite.id:<15}: {coefficient:>25}  Name: {metabolite.name:<60}  Charge: {metabolite.charge:>3}  Formula: {metabolite.formula}")
    sum += coefficient * metabolite.charge
print(f'\nSum charge: {sum}')
print("\nAssociated Genes:")
for gene in reaction.genes:
    print(gene.id)

Reaction ID: Growth
Name: Biomass reaction
Equation: 0.000223 10fthf_c + 0.000223 2dmmql8_c + 0.000223 5mthf_c + 0.000279 accoa_c + 0.513689 ala__L_c + 0.000223 amet_c + 0.030016 arab__L_c + 0.295792 arg__L_c + 0.241055 asn__L_c + 0.241055 asp__L_c + 29.0 atp_c + 0.005205 ca2_c + 0.000223 chor_c + 0.005205 cl_c + 0.002944 clpn160_p + 0.00229 clpn161_p + 0.00118 clpn181_p + 0.000576 coa_c + 0.0001 cobalt2_c + 0.133508 ctp_c + 0.000709 cu2_c + 0.09158 cys__L_c + 0.026166 datp_c + 0.027017 dctp_c + 0.027017 dgtp_c + 0.026166 dttp_c + 0.000223 fad_c + 0.006715 fe2_c + 0.007808 fe3_c + 0.644556 fru_c + 0.005772 gal_c + 0.117366841601577 glc__D_c + 0.26316 gln__L_c + 0.26316 glu__L_c + 0.612638 gly_c + 0.215096 gtp_c + 29.0 h2o_c + 0.000223 hemeO_c + 0.049935 hexadecacid_c + 0.007033 hexedecacid_c + 0.094738 his__L_c + 0.290529 ile__L_c + 0.195193 k_c + 0.019456 kdo2lipid4_p + 0.450531 leu__L_c + 0.005903 lnlc_c + 0.000364 lnlncg_c + 0.343161 lys__L_c + 3.1e-05 malcoa_c + 0.00481 man_c + 0.1

## Oxidative Phosphorylation complex addition

In [1]:
# we set an unlimited bound for the limiting substrate
reaction = model.reactions.get_by_id('EX_co2_e')
reaction.bounds = -100.0, 0.0

NameError: name 'model' is not defined

In [ ]:

reaction = model.reactions.get_by_id('EX_o2_e')
reaction.bounds = -6.92, 0.0


Addition of proton translocation to the NADH dehydrogenase reaction


In [ ]:
reaction = model.reactions.get_by_id('NADH5')
metabolite_to_add = 'h_p'
reaction.add_metabolites({metabolite_to_add: 4.0})

print(f"Reaction ID: {reaction.id}")
print(f"Name: {reaction.name}")
print(f"Equation: {reaction.reaction}")
print(f"Lower Bound: {reaction.lower_bound}")
print(f"Upper Bound: {reaction.upper_bound}")
print(reaction.annotation)
print(f"Genes: {[gene.id for gene in reaction.genes]}")


Reaction ID: NADH5
Name: NADH dehydrogenase (ubiquinone-8 )
Equation: 5.0 h_c + nadh_c + q8_c --> 4.0 h_p + nad_c + q8h2_c
Lower Bound: 0.0
Upper Bound: 1000.0
{'sbo': 'SBO:0000176', 'rhea': ['29107', '29108', '29109', '29110'], 'metanetx.reaction': 'MNXR101872', 'seed.reaction': 'rxn08975'}
Genes: ['AAFOLC_07640']


In [ ]:
summary = model.metabolites.get_by_id('h_p').summary()
summary

Percent,Flux,Reaction,Definition
0.00%,3.333E-05,CYTBD2pp,2.0 h_c + mql8_c + 0.5 o2_c --> h2o_c + 2.0 h_p + mqn8_c
33.24%,27.69,CYTBDpp,2.0 h_c + 0.5 o2_c + q8h2_c --> h2o_c + 2.0 h_p + q8_c
66.52%,55.41,NADH5,5.0 h_c + nadh_c + q8_c --> 4.0 h_p + nad_c + q8h2_c
0.23%,0.1951,PLIPA1G160pp,h2o_p + pg160_p --> 2agpg160_p + h_p + hdca_p
Percent,Flux,Reaction,Definition
99.77%,-83.11,ATPS4rpp,adp_c + 4.0 h_p + pi_c <=> atp_c + h2o_c + 3.0 h_c
0.23%,-0.1951,FACOAL160t2pp,atp_c + coa_c + h_p + hdca_p --> amp_c + h_c + pmtcoa_c + ppi_c


In [ ]:
model.slim_optimize()

0.024017630944258673

In [ ]:
model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.0002507,0,0.00%
cl_e,EX_cl_e,0.0002507,0,0.00%
co2_e,EX_co2_e,2.444,1,100.00%
cobalt2_e,EX_cobalt2_e,4.816E-06,0,0.00%
cu2_e,EX_cu2_e,3.415E-05,0,0.00%
fe2_e,EX_fe2_e,0.0007317,0,0.00%
h2_e,EX_h2_e,18.98,0,0.00%
k_e,EX_k_e,0.009401,0,0.00%
mg2_e,EX_mg2_e,0.0004178,0,0.00%
mn2_e,EX_mn2_e,3.328E-05,0,0.00%


# Setting the GAM value

In [18]:
GAM_metabolites_1 = {'atp_c': -150.5, 'h2o_c': -150.5}
GAM_metabolites_2 = {'adp_c': 150.5, 'pi_c': 150.5, 'h_c': 150.5}

# Get the biomass reaction
reaction = model.reactions.get_by_id('Growth')

# Update the coefficients for GAM_metabolites_1
for element, new_coefficient in GAM_metabolites_1.items():
    metabolite = model.metabolites.get_by_id(element)
    if metabolite in reaction.metabolites:
        print(f'Old coefficient for {element}: {reaction.metabolites[metabolite]}')
        reaction.add_metabolites({metabolite: new_coefficient - reaction.metabolites[metabolite]})
        print(f'Updated {element} coefficient to {new_coefficient}')

# Update the coefficients for GAM_metabolites_2
for element, new_coefficient in GAM_metabolites_2.items():
    metabolite = model.metabolites.get_by_id(element)
    if metabolite in reaction.metabolites:
        print(f'Old coefficient for {element}: {reaction.metabolites[metabolite]}')
        reaction.add_metabolites({metabolite: new_coefficient - reaction.metabolites[metabolite]})
        print(f'Updated {element} coefficient to {new_coefficient}')

# Optimize the model
solution = model.slim_optimize()
print(f'Optimal solution: {solution}')

Old coefficient for atp_c: -95.5
Updated atp_c coefficient to -150.5
Old coefficient for h2o_c: -95.5
Updated h2o_c coefficient to -150.5
Old coefficient for adp_c: 95.5
Updated adp_c coefficient to 150.5
Old coefficient for pi_c: 95.5
Updated pi_c coefficient to 150.5
Old coefficient for h_c: 95.5
Updated h_c coefficient to 150.5
Optimal solution: 0.05382834841324691


Set the NGAM value

In [ ]:
reaction = model.reactions.get_by_id('ATPM')
reaction.bounds = 9.27, 1000.0

In [1]:
sbml_filename = "C:\\Users\\adam-1y9pxi6q95ggpmx\\Documents\\PhD\\10-19 Research\\11 Data\\11.09 Models\\Manual_curation\\Energy_consumption\\241209_NGAM_coef_update_GAM.sbml"
cobra.io.write_sbml_model(model, sbml_filename)

NameError: name 'cobra' is not defined